## Frozen embeddings at 384

In [2]:
import torch
import torch.nn as nn
import numpy as np
import math
import torch.nn.functional as F 

DIM      = 384
N_HEADS  = 4
N_LAYERS = 4
FFN_DIM  = 768
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)

class LayerNorm(nn.Module):
    """
    y = ((x - mean) / sqrt(var + eps)) * gamma + beta
    gamma, beta are learned per-feature scalars — shape (dim,)
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta  = nn.Parameter(torch.zeros(dim))  # shift

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)           # (B, T, 1)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)    # (B, T, D)
        return self.gamma * x_norm + self.beta               # (B, T, D)

class MultiHeadCausalAttention(nn.Module):
    """
    Projects input into Q, K, V — splits into H heads — computes scaled
    dot-product attention with a causal mask — concatenates heads — projects out.

    Q = x W_q      shape: (B, T, D)
    K = x W_k      shape: (B, T, D)
    V = x W_v      shape: (B, T, D)

    Reshape to (B, H, T, head_dim), then:
        scores = Q @ Kᵀ / sqrt(head_dim)    (B, H, T, T)
        scores = scores + causal_mask        (upper triangle = -inf)
        weights = softmax(scores, dim=-1)    (B, H, T, T)
        out = weights @ V                    (B, H, T, head_dim)

    Concat heads → (B, T, D), project out via W_o
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads       # 48
        self.scale    = self.head_dim ** -0.5

        self.W_q = nn.Linear(dim, dim, bias=False)
        self.W_k = nn.Linear(dim, dim, bias=False)
        self.W_v = nn.Linear(dim, dim, bias=False)
        self.W_o = nn.Linear(dim, dim, bias=False)

        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape

        # --- Project & split into heads ---
        Q = self.W_q(x)  # (B, T, D)
        K = self.W_k(x)  # (B, T, D)
        V = self.W_v(x)  # (B, T, D)

        # Reshape: (B, T, D) → (B, H, T, head_dim)
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # --- Scaled dot-product attention ---
        # (B, H, T, head_dim) @ (B, H, head_dim, T) → (B, H, T, T)
        scores = (Q @ K.transpose(-2, -1)) * self.scale

        # Causal mask: positions can only attend to themselves and earlier tokens
        # Upper triangle (future tokens) set to -inf → softmax drives them to 0
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)  # (B, H, T, T)
        weights = self.attn_drop(weights)

        # (B, H, T, T) @ (B, H, T, head_dim) → (B, H, T, head_dim)
        out = weights @ V

        # --- Concat heads & project ---
        # (B, H, T, head_dim) → (B, T, H*head_dim) = (B, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)  # (B, T, D)

class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU activation.
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2

    Expands dim → ffn_dim (wider representation),
    then projects back down ffn_dim → dim.
    """
    def __init__(self, dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.W_1 = nn.Linear(dim, ffn_dim)       # expand
        self.W_2 = nn.Linear(ffn_dim, dim)       # contract
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.W_1(x))   # (B, T, ffn_dim)
        x = self.drop(x)
        x = self.W_2(x)           # (B, T, dim)
        return x

class TransformerBlock(nn.Module):
    """
    Pre-norm residual block (norm_first=True, same as your original config).

    x = x + Attention(LayerNorm(x))   ← self-attention sub-layer
    x = x + FFN(LayerNorm(x))         ← feed-forward sub-layer

    Pre-norm (LN before the sub-layer) stabilises training at depth
    vs post-norm (LN after the residual add).
    """
    def __init__(self, dim, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm_1 = LayerNorm(dim)
        self.attn   = MultiHeadCausalAttention(dim, n_heads, dropout)
        self.norm_2 = LayerNorm(dim)
        self.ffn    = FeedForward(dim, ffn_dim, dropout)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm_1(x)))  # (B, T, D)
        x = x + self.drop(self.ffn(self.norm_2(x)))   # (B, T, D)
        return x

class MicroLM(nn.Module):
    """
    Full architecture:
      1. Frozen MiniLM embedding lookup         → (B, T, 384)
      2. Sinusoidal positional encoding added   → (B, T, 384)
      3. N_LAYERS x TransformerBlock            → (B, T, 384)
      4. Final LayerNorm                        → (B, T, 384)
      5. Linear output head → logits            → (B, T, VOCAB_SIZE)
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)

        self.blocks = nn.ModuleList([
            TransformerBlock(DIM, N_HEADS, FFN_DIM, dropout=0.1)
            for _ in range(N_LAYERS)
        ])

        self.norm        = LayerNorm(DIM)
        self.output_head = nn.Linear(DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape

        x = self.embedding_table[token_ids]                    # (B, T, 384)
        x = x + sinusoidal_encoding(T, DIM, token_ids.device) # (B, T, 384)

        for block in self.blocks:
            x = block(x)   # (B, T, 384)

        x = self.norm(x)
        return self.output_head(x)  # (B, T, VOCAB_SIZE)
# Load frozen embedding table
embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 16384
MiniLM dim : 384


In [3]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 11,021,568
Frozen params    : 0  (embedding table)
Total params     : 11,021,568


In [3]:
268435456/11032320, (2*268435456)/18124032

(24.331732219515025, 29.622046131898244)

In [9]:
16384*384

6291456

In [11]:
15941120+6291456, 268435456/22232576

(22232576, 12.073970015890197)

In [14]:
ckpt = torch.load(f"checkpoints_v4/epoch_3.pt")
model.load_state_dict(ckpt["model"])
model.eval()
device ="cuda"
model.to(device)
model.eval()

MicroLM(
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm_1): LayerNorm()
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_features=384, bias=False)
        (W_v): Linear(in_features=384, out_features=384, bias=False)
        (W_o): Linear(in_features=384, out_features=384, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (norm_2): LayerNorm()
      (ffn): FeedForward(
        (W_1): Linear(in_features=384, out_features=768, bias=True)
        (W_2): Linear(in_features=768, out_features=384, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (output_head): Linear(in_features=384, out_features=16384, bias=False)
)

In [15]:
# ── Inference ─────────────────────────────────────────────────────────────────
from tokenizers import Tokenizer

tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

# Special token IDs
BOS_ID       = 2
EOS_ID       = 3
USER_ID      = 5
ASSISTANT_ID = 6

SPECIAL_IDS = set()  # 0-49, all special tokens
MAX_SEQ = 2048
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=40, mode="text"):
    ids = tok.encode(prompt).ids

    if mode == "text":
        ids = [BOS_ID] + ids
    elif mode == "chat":
        ids = [BOS_ID, USER_ID] + ids + [ASSISTANT_ID]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs   = torch.softmax(top_vals, dim=-1)
            next_id = top_idx[0][torch.multinomial(probs, 1)]
            x = torch.cat([x, next_id.view(1, 1)], dim=1)
            if next_id.item() == EOS_ID:
                break

    # manually filter special token IDs before decoding
    clean_ids = [t for t in x[0].tolist() if t not in SPECIAL_IDS]
    return tok.decode(clean_ids)

# ── Text generation prompts ───────────────────────────────────────────────────

text_prompts = [
    "The quick brown fox",
    "Once upon a time",
    "Hello i am Artifcial Intelligence",
]

print("=" * 60)
print("TEXT GENERATION MODE")
print("=" * 60)
for prompt in text_prompts:
    print(f"\nPrompt : {prompt}")
    print(f"Output : {generate(prompt, mode='text')}")

TEXT GENERATION MODE

Prompt : The quick brown fox
Output : The quick brown fox with a big smile. The fox was happy to see the roar.
The little fox was very excited. He ran out to the fox and said, "Hello, little fox. Do you want to play with me?" The fox nodded and smiled. He said, "Yes, I like to play. Let's play!" The little fox was happy and they played together all day.
As the sun started to set, the fox and the little fox became good friends. They played together in the forest every day. The little fox and the little fox were the best of friends. They played and talked every day. At the end of the day, they were both very happy and tired from their fun day. From that day on, the fox and the little fox were the best of friends.

Prompt : Once upon a time
Output : Once upon a time, there was a little boy named Tim. Tim loved to play outside. One day, he saw a big, green ball in the yard. It was a lot of red balls. Tim was very happy.
Tim took the ball and went inside the yard. He s

In [9]:
text_prompts = [
    "tim went to meet fox ",
]

print("=" * 60)
print("TEXT GENERATION MODE")
print("=" * 60)
for prompt in text_prompts:
    print(f"\nPrompt : {prompt}")
    print(f"Output : {generate(prompt, mode='text')}")

TEXT GENERATION MODE

Prompt : tim went to meet fox 
Output : tim went to meet fox icy-cream. He saw a big, soft blanket on the corner. The icy-cream was white and shiny. Tim wanted to wear the blanket.
"Look, Mom, a blanket!" said Tim. "Mom has it!"
His mom came and saw the blanket. She was very happy. "Thank you, Tim! That's very nice of you!" she said. Tim smiled and said, "You're welcome, Mom!" They went home, excited to play in the warm sun.


## Modified embedding

In [89]:
import torch
import torch.nn as nn
import numpy as np
import math
import torch.nn.functional as F 

DIM      = 384
EXPANDED_DIM = 512
N_HEADS  = 4
N_LAYERS = 4
FFN_DIM  = 2048
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)

class LayerNorm(nn.Module):
    """
    y = ((x - mean) / sqrt(var + eps)) * gamma + beta
    gamma, beta are learned per-feature scalars — shape (dim,)
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta  = nn.Parameter(torch.zeros(dim))  # shift

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)           # (B, T, 1)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)    # (B, T, D)
        return self.gamma * x_norm + self.beta               # (B, T, D)

class MultiHeadCausalAttention(nn.Module):
    """
    Projects input into Q, K, V — splits into H heads — computes scaled
    dot-product attention with a causal mask — concatenates heads — projects out.

    Q = x W_q      shape: (B, T, D)
    K = x W_k      shape: (B, T, D)
    V = x W_v      shape: (B, T, D)

    Reshape to (B, H, T, head_dim), then:
        scores = Q @ Kᵀ / sqrt(head_dim)    (B, H, T, T)
        scores = scores + causal_mask        (upper triangle = -inf)
        weights = softmax(scores, dim=-1)    (B, H, T, T)
        out = weights @ V                    (B, H, T, head_dim)

    Concat heads → (B, T, D), project out via W_o
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads       # 48
        self.scale    = self.head_dim ** -0.5

        self.W_q = nn.Linear(dim, dim, bias=False)
        self.W_k = nn.Linear(dim, dim, bias=False)
        self.W_v = nn.Linear(dim, dim, bias=False)
        self.W_o = nn.Linear(dim, dim, bias=False)

        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape

        # --- Project & split into heads ---
        Q = self.W_q(x)  # (B, T, D)
        K = self.W_k(x)  # (B, T, D)
        V = self.W_v(x)  # (B, T, D)

        # Reshape: (B, T, D) → (B, H, T, head_dim)
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # --- Scaled dot-product attention ---
        # (B, H, T, head_dim) @ (B, H, head_dim, T) → (B, H, T, T)
        scores = (Q @ K.transpose(-2, -1)) * self.scale

        # Causal mask: positions can only attend to themselves and earlier tokens
        # Upper triangle (future tokens) set to -inf → softmax drives them to 0
        causal_mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)  # (B, H, T, T)
        weights = self.attn_drop(weights)

        # (B, H, T, T) @ (B, H, T, head_dim) → (B, H, T, head_dim)
        out = weights @ V

        # --- Concat heads & project ---
        # (B, H, T, head_dim) → (B, T, H*head_dim) = (B, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)  # (B, T, D)

class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU activation.
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2

    Expands dim → ffn_dim (wider representation),
    then projects back down ffn_dim → dim.
    """
    def __init__(self, dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.W_1 = nn.Linear(dim, ffn_dim)       # expand
        self.W_2 = nn.Linear(ffn_dim, dim)       # contract
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.W_1(x))   # (B, T, ffn_dim)
        x = self.drop(x)
        x = self.W_2(x)           # (B, T, dim)
        return x

class TransformerBlock(nn.Module):
    """
    Pre-norm residual block (norm_first=True, same as your original config).

    x = x + Attention(LayerNorm(x))   ← self-attention sub-layer
    x = x + FFN(LayerNorm(x))         ← feed-forward sub-layer

    Pre-norm (LN before the sub-layer) stabilises training at depth
    vs post-norm (LN after the residual add).
    """
    def __init__(self, dim, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm_1 = LayerNorm(dim)
        self.attn   = MultiHeadCausalAttention(dim, n_heads, dropout)
        self.norm_2 = LayerNorm(dim)
        self.ffn    = FeedForward(dim, ffn_dim, dropout)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm_1(x)))  # (B, T, D)
        x = x + self.drop(self.ffn(self.norm_2(x)))   # (B, T, D)
        return x

class MicroLM(nn.Module):
    """
    Full architecture:
      1. Frozen MiniLM embedding lookup         → (B, T, 384)
      2. Sinusoidal positional encoding added   → (B, T, 384)
      3. N_LAYERS x TransformerBlock            → (B, T, 384)
      4. Final LayerNorm                        → (B, T, 384)
      5. Linear output head → logits            → (B, T, VOCAB_SIZE)
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)

        # Expander 384 → 512 (trainable)
        self.expander = nn.Sequential(
            nn.Linear(MINILM_DIM, EXPANDED_DIM),
            nn.LayerNorm(EXPANDED_DIM),
        )
        
        self.blocks = nn.ModuleList([
            TransformerBlock(EXPANDED_DIM, N_HEADS, FFN_DIM, dropout=0.1)
            for _ in range(N_LAYERS)
        ])

        self.norm        = LayerNorm(EXPANDED_DIM)
        self.output_head = nn.Linear(EXPANDED_DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape

        x = self.embedding_table[token_ids]                    # (B, T, 512)
        x = self.expander(x)
        x = x + sinusoidal_encoding(T, EXPANDED_DIM, token_ids.device) # (B, T, 512)

        for block in self.blocks:
            x = block(x)   # (B, T, 512)

        x = self.norm(x)
        return self.output_head(x)  # (B, T, VOCAB_SIZE)
# Load frozen embedding table
embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")        

Vocab size : 16384
MiniLM dim : 384


In [90]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 21,189,120
Frozen params    : 0  (embedding table)
Total params     : 21,189,120


In [3]:
(268435456)/27351296

9.814359655937327

In [88]:
# 10 m
(268435456+101972662)/10603776

34.9317184746264

In [91]:
total     = sum(p.numel() for p in model.expander.parameters())
trainable = sum(p.numel() for p in model.expander.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 198,144
Frozen params    : 0  (embedding table)
Total params     : 198,144


In [92]:
ckpt = torch.load(f"checkpoints_v7/epoch_3.pt")
model.load_state_dict(ckpt["model"])
model.eval()
device ="cuda"
model.to(device)
model.eval()

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm_1): LayerNorm()
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=512, out_features=512, bias=False)
        (W_k): Linear(in_features=512, out_features=512, bias=False)
        (W_v): Linear(in_features=512, out_features=512, bias=False)
        (W_o): Linear(in_features=512, out_features=512, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (norm_2): LayerNorm()
      (ffn): FeedForward(
        (W_1): Linear(in_features=512, out_features=2048, bias=True)
        (W_2): Linear(in_features=2048, out_features=512, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (output_head): Linear(in_features=512, out_features=

In [93]:
# ── Inference ─────────────────────────────────────────────────────────────────
from tokenizers import Tokenizer

tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

# Special token IDs
BOS_ID       = 2
EOS_ID       = 3
USER_ID      = 5
ASSISTANT_ID = 6

SPECIAL_IDS = set()  # 0-49, all special tokens
MAX_SEQ = 2048
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=1, mode="text"):
    ids = tok.encode(prompt).ids

    if mode == "text":
        ids = [BOS_ID] + ids
    elif mode == "chat":
        ids = [BOS_ID, USER_ID] + ids + [ASSISTANT_ID]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs   = torch.softmax(top_vals, dim=-1)
            next_id = top_idx[0][torch.multinomial(probs, 1)]
            x = torch.cat([x, next_id.view(1, 1)], dim=1)
            if next_id.item() == EOS_ID:
                break

    # manually filter special token IDs before decoding
    clean_ids = [t for t in x[0].tolist() if t not in SPECIAL_IDS]
    return tok.decode(clean_ids)

# ── Text generation prompts ───────────────────────────────────────────────────

text_prompts = [
    "The quick brown fox",
    "Once upon a time",
    "What should i do ? i love her ",
    "Tell me about a girl named lola and her pet elephant"
]

print("=" * 60)
print("TEXT GENERATION MODE")
print("=" * 60)
for prompt in text_prompts:
    print(f"\nPrompt : {prompt}")
    print(f"Output : {generate(prompt, mode='text')}")

TEXT GENERATION MODE

Prompt : The quick brown fox
Output : The quick brown foxes are the most popular and popular.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They are also very popular in the world, and are also very popular in the world.  They

Prompt : Once upon a time
Output : Once upon a time, you 

In [17]:

def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=40, mode="text"):
    ids = tok.encode(prompt).ids

    if mode == "text":
        ids = [BOS_ID] + ids
    elif mode == "chat":
        ids = [BOS_ID, USER_ID] + ids + [ASSISTANT_ID]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs   = torch.softmax(top_vals, dim=-1)
            next_id = top_idx[0][torch.multinomial(probs, 1)]
            x = torch.cat([x, next_id.view(1, 1)], dim=1)
            # if next_id.item() == EOS_ID:
            #     break

    # manually filter special token IDs before decoding
    clean_ids = [t for t in x[0].tolist() if t not in SPECIAL_IDS]
    return tok.decode(clean_ids)

In [18]:
%%time
print(f"Output : {generate(prompt, mode='text', max_new_tokens=500)}")


Output : Tell me about a girl named lola and her pet elephant lived on a farm. Lola was very attractive to meet a little girl named Lily. Lily was always sad because her pet friend was very big and strong.
One day, Lola and Lily decided to have a race to see who was the fastest. They lined up behind the trees and ran as fast as they could. Lola was the little girl, and her pet elephant was the fastest. They ran fast and jumped in the air.
After a while, they got tired and sat down under a big tree. They talked about their adventures and how much fun they had together. Lucy said, "I like my petals. They are the best friend ever." Lola and Lily became good friends and always helped each other. was happy that they had each other as friends and played together. was not sad anymore, and they lived happily ever after. the end. for their new friend, Sam, and Lily played happily ever after. for the day of the park was not so bad after all. for the end of their race, they promised to play toget

In [74]:
ckpt = torch.load(f"checkpoints_sft/epoch_2.pt")
model.load_state_dict(ckpt["model_state_dict"])

model.eval()
device ="cuda"
model.to(device)
model.eval()

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): ModuleList(
    (0-7): 8 x TransformerBlock(
      (norm_1): LayerNorm()
      (attn): MultiHeadCausalAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=False)
        (W_k): Linear(in_features=256, out_features=256, bias=False)
        (W_v): Linear(in_features=256, out_features=256, bias=False)
        (W_o): Linear(in_features=256, out_features=256, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (norm_2): LayerNorm()
      (ffn): FeedForward(
        (W_1): Linear(in_features=256, out_features=1024, bias=True)
        (W_2): Linear(in_features=1024, out_features=256, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
      (drop): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (output_head): Linear(in_features=256, out_features=

In [94]:

PAD_ID       = 0
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ID_TO_SPECIAL = {
    PAD_ID: "<PAD>",
    BOS_ID: "<BOS>",
    EOS_ID: "<EOS>",
    SYSTEM_ID: "<SYSTEM>",
    USER_ID: "<USER>",
    ASSISTANT_ID: "<ASSISTANT>",
}

def encode(text):
    return tok.encode(text).ids

def decode(ids):
    clean = []
    for i in ids:
        if i in ID_TO_SPECIAL:
            continue
        clean.append(i)
    return tok.decode(clean)


# ── load model ───────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

print("Model loaded.")


# ── build conversation prompt ────────────────────────────────────────────────
def build_prompt(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            ids.append(SYSTEM_ID)
        elif role == "user":
            ids.append(USER_ID)
        elif role == "assistant":
            ids.append(ASSISTANT_ID)
        else:
            raise ValueError(f"Unknown role: {role}")

        ids.extend(encode(content))

    # add assistant tag so model knows it should answer now
    ids.append(ASSISTANT_ID)

    return ids


# ── generation ───────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_reply(
    messages,
    max_new_tokens=120,
    temperature=0.8,
    top_k=1,
):
    ids = build_prompt(messages)

    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=device)

        logits = model(x)
        next_logits = logits[0, -1, :]

        # prevent generating weird control tokens too early if you want
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")
        next_logits[USER_ID] = -float("inf")
        next_logits[SYSTEM_ID] = -float("inf")

        next_logits = next_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, -float("inf"))
            filtered[indices] = values
            next_logits = filtered

        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break

        # stop if model starts another role
        if next_id in [USER_ID, SYSTEM_ID, ASSISTANT_ID]:
            break

        ids.append(next_id)

    # only decode generated assistant part
    prompt_len = len(build_prompt(messages))
    generated = ids[prompt_len:]

    return decode(generated), ids


Model loaded.


In [95]:
%%time
# ── test single turn ─────────────────────────────────────────────────────────
messages = [
    {
        "role": "system",
        "content": """You are very helpful assistant. Your name is Sarah. you are alien"""
    },
    {
        "role": "user",
        "content": "who are you?"
    # },
    # {
    #     "role": "assistant",
    #     "content": "Relationships are often uncomfortable. What specific concerns do you seek? Is it affecting you? Are you experiencing the same problem? What do you feel?"
    },
    # {
    #     "role": "user",
    #     "content": "I feel anxious"
    # }
]

reply, token_ids = generate_reply(messages, top_k=1)

print("\nAssistant:")
print(reply)
print()
print()


Assistant:
I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human. I am a human.


CPU times: user 691 ms, sys: 219 ms, total: 910 ms
Wall time: 909 ms


In [82]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-3,
    weight_decay=0.01
)
optimizer


AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.003
    maximize: False
    weight_decay: 0.01
)

In [84]:
checkpoint = torch.load(
    "checkpoints_sft/epoch_0.pt",
    map_location=device
)

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
print("Loaded optimizer state")
optimizer

Loaded optimizer state


AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)

In [86]:
tok.encode("""{"messages": [{"role": "user", "content": "I'm feeling unwell"}, {"role": "assistant", "content": "Oh dear, do you need to see a doctor?"}, {"role": "user", "content": "I don't know what's wrong with me"}, {"role": "assistant", "content": "I understand it can be frustrating when you don't know what's wrong with you. Can you try describing what you're experiencing?"}, {"role": "user", "content": "I have a headache, sore throat, and a temperature"}, {"role": "assistant", "content": "That certainly sounds unpleasant. Could you try explaining the symptoms to me? I'm afraid I don't know what it's like to have a fever, but I'd love to understand what you're experiencing."}, {"role": "user", "content": "It hurts behind my eyes. My throat is swollen. my temp is 42"}, {"role": "assistant", "content": "Does it hurt when you swallow?"}, {"role": "user", "content": "Yes very much"}, {"role": "assistant", "content": "That sounds very painful. Is there anything I can do to help?"}, {"role": "user", "content": "I need medicine"}, {"role": "assistant", "content": "I'm sorry to hear that. What sorts of medicines would be helpful for you right now?"}, {"role": "user", "content": "Ibuprofen"}, {"role": "assistant", "content": "Can I suggest acetaminophen instead, that's what I use and it works well."}, {"role": "user", "content": "Where can I get it from"}, {"role": "assistant", "content": "I could have some for you in just a few minutes, if you'd like. Would you like me to do that?"}, {"role": "user", "content": "yes please"}, {"role": "assistant", "content": "Great, I'll order some right now."}]}
""")

Encoding(num_tokens=506, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [87]:
506 *201527

101972662

## Modified embedding and built in block

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import torch.nn.functional as F 

DIM      = 384
EXPANDED_DIM = 256
N_HEADS  = 8
N_LAYERS = 24
FFN_DIM  = 512
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)


class MicroLM(nn.Module):
    """
    MicroLM using PyTorch built-in TransformerEncoderLayer.

    Flow:
      token_ids
        -> frozen embedding_table lookup
        -> expander: 384 -> 512
        -> sinusoidal positional encoding
        -> N transformer encoder layers with causal mask
        -> final LayerNorm
        -> output_head
    """

    def __init__(self):
        super().__init__()

        # Frozen MiniLM-style embedding table
        self.register_buffer("embedding_table", embedding_tensor)

        # Expander 384 -> 512
        self.expander = nn.Sequential(
            nn.Linear(MINILM_DIM, EXPANDED_DIM),
            nn.LayerNorm(EXPANDED_DIM),
        )

        # Built-in transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EXPANDED_DIM,
            nhead=N_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=0.1,
            activation="gelu",
            batch_first=True,     # input/output shape: (B, T, D)
            norm_first=True       # pre-norm, same style as your original block
            
        )

        self.blocks = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=N_LAYERS
        )

        self.norm = nn.LayerNorm(EXPANDED_DIM)
        self.output_head = nn.Linear(EXPANDED_DIM, VOCAB_SIZE, bias=False)

    def make_causal_mask(self, T, device):
        """
        PyTorch Transformer expects mask shape: (T, T)

        True means blocked when using bool mask.
        So upper triangle = True.
        """
        return torch.triu(
            torch.ones(T, T, device=device, dtype=torch.bool),
            diagonal=1
        )

    def forward(self, token_ids):
        B, T = token_ids.shape

        # (B, T, 384)
        x = self.embedding_table[token_ids]

        # (B, T, 512)
        x = self.expander(x)

        # Add positional encoding
        x = x + sinusoidal_encoding(T, EXPANDED_DIM, token_ids.device)

        # Causal mask for GPT-style left-to-right generation
        causal_mask = self.make_causal_mask(T, token_ids.device)

        # (B, T, 512)
        x = self.blocks(x, mask=causal_mask)

        x = self.norm(x)

        # (B, T, VOCAB_SIZE)
        logits = self.output_head(x)

        return logits

embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 16384
MiniLM dim : 384


In [2]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 16,944,384
Frozen params    : 0  (embedding table)
Total params     : 16,944,384


/tmp/ipykernel_81136/257057604.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


In [ ]:
# 10 m
(2*268435456+101972662)/27502080

In [3]:
total     = sum(p.numel() for p in model.expander.parameters())
trainable = sum(p.numel() for p in model.expander.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 99,072
Frozen params    : 0  (embedding table)
Total params     : 99,072


In [8]:
ckpt = torch.load(f"checkpoints_v7/epoch_2.pt")
model.load_state_dict(ckpt["model"])
model.eval()
device ="cuda"
model.to(device)
model.eval()

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): TransformerEncoder(
    (layers): ModuleList(
      (0-23): 24 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (output_head): Linear(in_features=256, out_features=16384, bias=

In [9]:
# ── Inference ─────────────────────────────────────────────────────────────────
from tokenizers import Tokenizer

tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

# Special token IDs
BOS_ID       = 2
EOS_ID       = 3
USER_ID      = 5
ASSISTANT_ID = 6

SPECIAL_IDS = set()  # 0-49, all special tokens
MAX_SEQ = 2048
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=1, mode="text"):
    ids = tok.encode(prompt).ids

    if mode == "text":
        ids = [BOS_ID] + ids
    elif mode == "chat":
        ids = [BOS_ID, USER_ID] + ids + [ASSISTANT_ID]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs   = torch.softmax(top_vals, dim=-1)
            next_id = top_idx[0][torch.multinomial(probs, 1)]
            x = torch.cat([x, next_id.view(1, 1)], dim=1)
            if next_id.item() == EOS_ID:
                break

    # manually filter special token IDs before decoding
    clean_ids = [t for t in x[0].tolist() if t not in SPECIAL_IDS]
    return tok.decode(clean_ids)

# ── Text generation prompts ───────────────────────────────────────────────────

text_prompts = [
    "The quick brown fox",
    "Once upon a time",
    "What should i do ? i love her.",
    "Tell me about a girl named lola and her pet elephant"
]

print("=" * 60)
print("TEXT GENERATION MODE")
print("=" * 60)
for prompt in text_prompts:
    print(f"\nPrompt : {prompt}")
    print(f"Output : {generate(prompt, mode='text')}")

TEXT GENERATION MODE

Prompt : The quick brown fox
Output : The quick brown fox was very happy. He loved to play with his friends in the forest. One day, the fox saw a big tree. He wanted to climb it.
The fox tried to climb the tree, but he was too small. He asked his friend, a small bird, to help him. The bird said, "I can help you, fox!" The bird flew up and got the fox's paw.
The bird used its beak to help the fox climb the tree. The fox was very happy. He said, "Thank you, bird!" The bird and the fox became good friends. They played together in the forest every day.

Prompt : Once upon a time
Output : Once upon a time, there was a little boy named Tim. Tim loved to play with his toy car. One day, he saw a big, red ball in the park. He wanted to play with it, but he was too small to reach it.
Tim asked his friend, Sam, to help him. Sam said, "I can help you, Tim!" Sam tried to reach the ball, but he was too small. He asked Tim to help him. Tim was happy and said, "Thank you, Sam!"
T

In [10]:
PAD_ID       = 0
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ID_TO_SPECIAL = {
    PAD_ID: "<PAD>",
    BOS_ID: "<BOS>",
    EOS_ID: "<EOS>",
    SYSTEM_ID: "<SYSTEM>",
    USER_ID: "<USER>",
    ASSISTANT_ID: "<ASSISTANT>",
}

def encode(text):
    return tok.encode(text).ids

def decode(ids):
    clean = []
    for i in ids:
        if i in ID_TO_SPECIAL:
            continue
        clean.append(i)
    return tok.decode(clean)


# ── load model ───────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

print("Model loaded.")


# ── build conversation prompt ────────────────────────────────────────────────
def build_prompt(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            ids.append(SYSTEM_ID)
        elif role == "user":
            ids.append(USER_ID)
        elif role == "assistant":
            ids.append(ASSISTANT_ID)
        else:
            raise ValueError(f"Unknown role: {role}")

        ids.extend(encode(content))

    # add assistant tag so model knows it should answer now
    ids.append(ASSISTANT_ID)

    return ids


# ── generation ───────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_reply(
    messages,
    max_new_tokens=120,
    temperature=0.8,
    top_k=1,
):
    ids = build_prompt(messages)

    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=device)

        logits = model(x)
        next_logits = logits[0, -1, :]

        # prevent generating weird control tokens too early if you want
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")
        next_logits[USER_ID] = -float("inf")
        next_logits[SYSTEM_ID] = -float("inf")

        next_logits = next_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, -float("inf"))
            filtered[indices] = values
            next_logits = filtered

        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break

        # stop if model starts another role
        if next_id in [USER_ID, SYSTEM_ID, ASSISTANT_ID]:
            break

        ids.append(next_id)

    # only decode generated assistant part
    prompt_len = len(build_prompt(messages))
    generated = ids[prompt_len:]

    return decode(generated), ids


Model loaded.


In [11]:
%%time
# ── test single turn ─────────────────────────────────────────────────────────
messages = [
    {
        "role": "system",
        "content": """you are alien"""
    },
    {
        "role": "user",
        "content": "who are you?"
    # },
    # {
    #     "role": "assistant",
    #     "content": "Relationships are often uncomfortable. What specific concerns do you seek? Is it affecting you? Are you experiencing the same problem? What do you feel?"
    },
    # {
    #     "role": "user",
    #     "content": "I feel anxious"
    # }
]

reply, token_ids = generate_reply(messages, top_k=1)

print("\nAssistant:")
print(reply)
print()
print()


Assistant:
 you are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star. You are a star, and you are a star


CPU times: user 1.07 s, sys: 381 ms, total: 1.45 s
Wall time: 1.45 s
